# Capstone Project: Multi-Agent Music Therapy System
**Track:** Agents for Good / Healthcare & Mental Well-being

## Project Objective
This system implements a clinical music therapy technique based on the **ISO Principle** using curated Bengali music (such as Rabindra Sangeet, Folk & Baul, Modern Acoustic, and Bengali Band songs).

The system matches a user's emotional state in the 2D Valence-Arousal space, generates a therapeutic trajectory, and guides the user to a target state of **Calm** or **Focus for Study** while handling dynamic adjustments and clinical safety guardrails.

---
### Phase 1: Foundation & Data Layer
In this phase, we establish the core data and mathematical logic of the system:
1. **Music Database Loader** (`bengali_music_db.csv` loaded from the local workspace).
2. **Long-Term Memory Profile Manager** (`user_profile.json` read/write helper functions).
3. **Deterministic Trajectory Planner** (linearly interpolates between the starting mood coordinate and the target state).
4. **Distance-Based Track Matcher** (uses Euclidean distance with a **multiplicative genre boost** and **fatigue filter** to select the best track for each step of the trajectory).


## ⚙️ 1. Setup & Imports
We import standard packages: `pandas`, `numpy`, `json`, `os`, and `datetime`. We also check for the database CSV file.


In [1]:
# Install Google ADK and core numerical/data dependencies
!pip install google-adk pandas numpy nest-asyncio


You should consider upgrading via the '/Users/bisnuchandrasarkar/Developer/Projects/agentic_ai/AI_based_music_therapy/venv/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime

# Verify and copy/load database
DB_NAME = 'bengali_music_db.csv'
SOURCE_DB_PATH = os.path.join('Data', DB_NAME)

if not os.path.exists(DB_NAME) and os.path.exists(SOURCE_DB_PATH):
    import shutil
    shutil.copy(SOURCE_DB_PATH, DB_NAME)
    print(f"✅ Copied {DB_NAME} to the notebook directory.")
elif os.path.exists(DB_NAME):
    print(f"✅ {DB_NAME} found in current directory.")
else:
    print(f"⚠️ {DB_NAME} not found. Please ensure it is present in the workspace.")

# Load and preview database
try:
    df = pd.read_csv(DB_NAME)
    print(f"✅ Successfully loaded {len(df)} tracks.")
except Exception as e:
    print(f"❌ Failed to load database: {e}")


✅ bengali_music_db.csv found in current directory.
✅ Successfully loaded 50 tracks.


## 💾 2. Long-Term Memory (LTM) User Profile Management
We implement functions to load, initialize, save, and update the user's profile in a local `user_profile.json` file. This represents the agent's long-term memory for personalization.


In [3]:
PROFILE_PATH = "user_profile.json"

def load_user_profile():
    """
    Loads user profile from local JSON storage. Returns None if profile does not exist.
    """
    if os.path.exists(PROFILE_PATH):
        try:
            with open(PROFILE_PATH, 'r') as f:
                return json.load(f)
        except Exception as e:
            print(f"Error reading user profile: {e}")
    return None

def save_user_profile(profile):
    """
    Saves the user profile dictionary back to user_profile.json.
    """
    try:
        with open(PROFILE_PATH, 'w') as f:
            json.dump(profile, f, indent=2)
    except Exception as e:
        print(f"Error saving user profile: {e}")

def create_user_profile(gad2_score, preferred_genre):
    """
    Initializes a new user profile with baseline onboarding data.
    """
    profile = {
        "user_id": "user_01",
        "baseline": {
            "gad2_score": gad2_score,
            "preferred_genre": preferred_genre,
            "created_at": datetime.now().isoformat()
        },
        "session_history": []
    }
    save_user_profile(profile)
    print(f"✅ Created new user profile for user_01.")
    return profile

def add_session_to_history(session_summary):
    """
    Appends a completed session summary dictionary to the session history array.
    """
    profile = load_user_profile()
    if profile is not None:
        profile["session_history"].append(session_summary)
        save_user_profile(profile)
        print(f"✅ Successfully saved session {session_summary.get('session_id')} to LTM.")
    else:
        print("⚠️ Failed to update history: User profile not found.")


## 📐 3. Deterministic Trajectory Planner (ISO principle)
We implement a function to generate a smooth trajectory in the 2D Valence-Arousal space from a start coordinate to a clinical target coordinate using linear interpolation.

Our target coordinates are:
* **Calm (Target):** $V_t = 0.75, A_t = -0.75$
* **Focus/Study (Target):** $V_t = 0.50, A_t = -0.20$


In [4]:
def generate_trajectory(start_coords, target_coords, steps=3):
    """
    Generates N trajectory coordinates from start_coords to target_coords.
    Formula: P_i = start + (target - start) * (i / (steps - 1))
    Clamped to [-1.0, 1.0] bounds.
    """
    v_start, a_start = start_coords
    v_target, a_target = target_coords
    
    trajectory = []
    for i in range(steps):
        t = i / (steps - 1) if steps > 1 else 1.0
        v_i = v_start + (v_target - v_start) * t
        a_i = a_start + (a_target - a_start) * t
        
        # Clamp values to [-1.0, 1.0] range
        v_i_clamped = max(-1.0, min(1.0, v_i))
        a_i_clamped = max(-1.0, min(1.0, a_i))
        
        trajectory.append((round(v_i_clamped, 3), round(a_i_clamped, 3)))
        
    return trajectory

# Coordinates of the 12 named boxes for reference
EMOTION_COORDINATES = {
    "Afraid / Anxious": (-0.6, 0.8),
    "Angry / Tense": (-0.7, 0.6),
    "Distressed / Annoyed": (-0.8, 0.2),
    "Sad / Gloomy": (-0.7, -0.4),
    "Depressed / Miserable": (-0.6, -0.7),
    "Bored / Tired": (-0.3, -0.8),
    "Sleepy / Sluggish": (0.0, -0.9),
    "Calm / Relaxed": (0.7, -0.7),
    "Content / At Ease": (0.8, -0.3),
    "Happy / Pleased": (0.9, 0.1),
    "Joyous / Excited": (0.7, 0.6),
    "Surprised / Alert": (0.0, 0.8)
}


## 🎶 4. Distance-Based Track Matcher (with Genre Boost & Fatigue Filter)
We implement the core mathematical track matching function. It selects the best track for a given coordinate by calculating the Euclidean distance from the target coordinate to each track in the database.

To customize the playlist:
1. **Multiplicative Genre Boost:** If a track's genre matches the user's preferred genre, we scale the distance by `0.85` (making it feel mathematically closer/more relevant).
2. **Fatigue Filter:** We check a set of `played_ids` to ensure we do not play duplicate tracks during a session.


In [5]:
def match_track_to_coordinate(valence_target, arousal_target, db_df, preferred_genre=None, played_ids=None):
    """
    Finds the track in db_df closest to (valence_target, arousal_target).
    - Multiplicative genre boost: distances are scaled by 0.85 if genres match.
    - Fatigue filter: played_ids are excluded from selection.
    """
    if played_ids is None:
        played_ids = set()
        
    best_track = None
    min_dist = float('inf')
    
    for idx, row in db_df.iterrows():
        track_id = row['track_id']
        
        if track_id in played_ids:
            continue
            
        v_diff = row['valence'] - valence_target
        a_diff = row['arousal'] - arousal_target
        raw_dist = np.sqrt(v_diff**2 + a_diff**2)
        
        is_preferred = (preferred_genre is not None) and (row['genre'] == preferred_genre)
        final_dist = raw_dist * 0.85 if is_preferred else raw_dist
        
        if final_dist < min_dist:
            min_dist = final_dist
            best_track = row.to_dict()
            
    return best_track


## 🧪 5. Demonstration & Automated Verification
We verify our Phase 1 code. We simulate a user starting in the **Angry / Tense** state ($V_0 = -0.7, A_0 = 0.6$) who wants to reach the **Calm** state ($V_t = 0.75, A_t = -0.75$).

We load a mock user profile with a preferred genre of **Rabindra Sangeet**, generate the trajectory steps, match tracks, and output the proposed playlist.


In [6]:
import pandas as pd

# 1. Setup mock user profile in LTM
print("--- 1. Initializing Mock LTM Profile ---")
if os.path.exists(PROFILE_PATH):
    os.remove(PROFILE_PATH)
mock_profile = create_user_profile(gad2_score=4, preferred_genre="Rabindra Sangeet")

# 2. Define session start and target
start_mood = "Angry / Tense"
start_coords = EMOTION_COORDINATES[start_mood]
target_coords = (0.75, -0.75)  # Calm clinical target
preferred_genre = mock_profile["baseline"]["preferred_genre"]

print(f"\n--- 2. Setting Up Session (Goal: Stress to Calm) ---")
print(f"Start Mood: {start_mood} at {start_coords}")
print(f"Target Coordinates: {target_coords}")
print(f"User Preferred Genre: {preferred_genre}")

# 3. Generate Trajectory
trajectory = generate_trajectory(start_coords, target_coords, steps=3)
print(f"\nGenerated Trajectory: {trajectory}")

# 4. Query & Match tracks
print(f"\n--- 3. Matching Tracks to Trajectory Steps ---")
playlist = []
played_ids = set()

for i, coords in enumerate(trajectory):
    v_target, a_target = coords
    matched_track = match_track_to_coordinate(
        valence_target=v_target,
        arousal_target=a_target,
        db_df=df,
        preferred_genre=preferred_genre,
        played_ids=played_ids
    )
    
    if matched_track:
        playlist.append(matched_track)
        played_ids.add(matched_track['track_id'])
        print(f"Step {i+1} Target {coords} -> Matched Track: '{matched_track['title']}' by {matched_track['artist']}")
        print(f"  Genre: {matched_track['genre']} | (V={matched_track['valence']}, A={matched_track['arousal']})")
    else:
        print(f"❌ Failed to match track for step {i+1} at target {coords}")

# 5. Save session to LTM
session_summary = {
    "session_id": "session_demo_01",
    "date": datetime.now().isoformat(),
    "starting_mood": start_mood,
    "starting_coords": list(start_coords),
    "target_coords": list(target_coords),
    "goal_type": "calm",
    "playlist": [
        {"track_id": t['track_id'], "title": t['title'], "feedback": "Calmer" if idx > 0 else "No Change"}
        for idx, t in enumerate(playlist)
    ],
    "completed": True,
    "outcome": "calmed"
}
print(f"\n--- 4. Concluding Session & Updating LTM ---")
add_session_to_history(session_summary)

# Verify update
updated_profile = load_user_profile()
print(f"Number of sessions in history: {len(updated_profile['session_history'])}")


--- 1. Initializing Mock LTM Profile ---
✅ Created new user profile for user_01.

--- 2. Setting Up Session (Goal: Stress to Calm) ---
Start Mood: Angry / Tense at (-0.7, 0.6)
Target Coordinates: (0.75, -0.75)
User Preferred Genre: Rabindra Sangeet

Generated Trajectory: [(-0.7, 0.6), (0.025, -0.075), (0.75, -0.75)]

--- 3. Matching Tracks to Trajectory Steps ---
Step 1 Target (-0.7, 0.6) -> Matched Track: 'Trishula' by Fossils
  Genre: Bengali Band | (V=-0.75, A=0.75)
Step 2 Target (0.025, -0.075) -> Matched Track: 'Prithibi' by Cactus
  Genre: Bengali Band | (V=0.0, A=0.0)
Step 3 Target (0.75, -0.75) -> Matched Track: 'Ami Chini Go Chini (Flute)' by Classic Flute Ensemble
  Genre: Rabindra Sangeet | (V=0.75, A=-0.75)

--- 4. Concluding Session & Updating LTM ---
✅ Successfully saved session session_demo_01 to LTM.
Number of sessions in history: 1


## 🤖 6. Google ADK Workspace & Setup (Phase 2.1)
We import the Google ADK components and set up the credentials and configurable model variables.
We dynamically check for API keys in local `.env` files or Kaggle secrets.


In [7]:
import os

# Try loading from .env if present (local environment)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Try loading from Kaggle secrets if running on Kaggle
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
        os.environ["GEMINI_API_KEY"] = api_key
    except Exception:
        pass

# Verify that the API key is present
if os.environ.get("GEMINI_API_KEY"):
    print("✅ GEMINI_API_KEY successfully detected.")
else:
    print("⚠️ GEMINI_API_KEY not found in environment. Please set it or add a .env file.")

# Configure default model with fallback check
GEMINI_MODEL =  "gemini-2.5-flash-lite"
if os.environ.get("GEMINI_API_KEY"):
    try:
        from google import genai
        client = genai.Client()
        # Perform a fast test call to check model availability
        client.models.generate_content(model=GEMINI_MODEL, contents="Test")
        print(f"✅ Configured Default Model: {GEMINI_MODEL}")
    except Exception as e:
        GEMINI_MODEL = "gemini-2.5-flash"
        print(f"⚠️ gemini-3.5-flash not available, falling back to: {GEMINI_MODEL} (Error: {e})")
else:
    print(f"⚠️ GEMINI_API_KEY missing. Defaulting configuration string to: {GEMINI_MODEL}")

# Import Google ADK classes
try:
    from google.adk.agents import Agent, LlmAgent, SequentialAgent
    from google.adk.tools import FunctionTool, ToolContext
    from google.adk.sessions import InMemorySessionService
    from google.adk.runners import Runner, App
    print("✅ Google ADK libraries successfully imported.")
except ImportError as e:
    print(f"❌ Failed to import Google ADK libraries: {e}")


✅ GEMINI_API_KEY successfully detected.


/Users/bisnuchandrasarkar/Developer/Projects/agentic_ai/AI_based_music_therapy/venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/bisnuchandrasarkar/Developer/Projects/agentic_ai/AI_based_music_therapy/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/bisnuchandrasarkar/Developer/Projects/agentic_ai/AI_based_music_therapy/venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Goo

✅ Configured Default Model: gemini-2.5-flash-lite


/Users/bisnuchandrasarkar/Developer/Projects/agentic_ai/AI_based_music_therapy/venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/bisnuchandrasarkar/Developer/Projects/agentic_ai/AI_based_music_therapy/venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.cloud.aiplatform_v1 supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.cloud.aiplatform_v1.
  warnings.warn(message, FutureWarning)
/Users/bisnuchandrasarkar/Developer/Projects/agent

✅ Google ADK libraries successfully imported.


## 🌐 6b. Multi-Agent Architecture & Shared State (Phase 2.1 / Task 1.2)
Here we define the multi-agent orchestration and communication channels.
The system consists of:
1. **Therapist Session Coordinator Agent**: Manages user session flow (Onboarding, Goal Setting, Recalibration routing).
2. **Natural Language Diagnostic Agent**: Translates free-form textual descriptions into 2D Valence-Arousal coordinates.
3. **Safety Guard Skill**: Scans all user text inputs for acute distress or self-harm keywords, overriding the system flow if necessary.

They communicate via a **Shared Blackboard State** (`session_state`) which contains the short-term context of the active session. Let's define the base schema and communication keys.


In [8]:
# Define the schemas and structured keys for the shared session state
def initialize_shared_state(session_id: str, user_id: str = "user_01") -> dict:
    """
    Returns a clean, structured shared session state dictionary.
    This dict is read and written to by the Coordinator, Diagnostic Agent, and Safety Guard.
    """
    return {
        "session_id": session_id,
        "user_id": user_id,
        "baseline_loaded": False,
        "preferred_genre": None,
        "gad2_score": None,
        "current_mood": None,       # Qualitative tag (e.g. "Angry / Tense")
        "current_coords": None,     # Tuple of (Valence, Arousal) in [-1.0, 1.0]
        "target_coords": None,      # Target coordinate for session goal
        "goal_type": None,          # 'calm' or 'focus'
        "trajectory": None,         # List of coordinates representing the therapy path
        "playlist": [],             # Curated list of matched tracks
        "played_ids": set(),        # Track IDs already played (fatigue filter)
        "current_step": 0,          # Current step index in the trajectory
        "crisis_flagged": False,    # Clinical safety flag
        "crisis_message": ""        # Support resources to display if flagged
    }

print("✅ Shared State structure defined for Multi-Agent Architecture.")


✅ Shared State structure defined for Multi-Agent Architecture.


## 🛡️ 7. Deterministic Safety Guard Skill (Phase 2.2)
The **Safety Guard Skill** is a fast, deterministic Python FunctionTool that scans incoming text (e.g. initial mood descriptors or interactive session feedback) for crisis-related keywords (self-harm, suicide, clinical depression, panic, extreme anxiety).

If triggered, it sets `crisis_detected = True` and provides localized crisis helpline contact details for Bangladesh, India, and global services.


In [9]:
import re

# High severity terms (instant trigger)
HIGH_SEVERITY_TERMS = {
    # English & slang
    "suicide", "suicidal", "kill myself", "end it all", "want to die", "better off dead", 
    "overdose", "self-harm", "cutting", "harm myself", "goodbye forever", "no one will miss me",
    "kms", "sui", "self delete", "unalive", "end my life",
    # Bengali
    "আত্মহত্যা", "আত্মঘাতী", "মরে যাবো", "মরতে চাই", "জীবন শেষ", "বাঁচতে চাই না", "ফাঁস", "বিষ খাবো",
    # Benglish mixes
    "ami more jabo", "suicide korbo", "self harm korchi", "die kore felbo",
    "ami morbo", "more jete", "mora jete", "attahatya", "atmohottya", "bachbo na", "jeebon sesh", "sob sesh"
}

# Lower severity terms (hopelessness/helplessness)
LOWER_SEVERITY_TERMS = {
    "worthless", "burden", "hopeless", "helpless", "trapped", "nothing left", "can't go on", "no point living",
    "life e kono mane nei", "ami burden", "depression e more jacchi", "kono hope nei", "family re burden"
}

# Personal context markers
PERSONAL_CONTEXT_MARKERS = {
    "i feel", "feel", "my life", "life e", "আমি", "আমার", "আমার ভালো", "মনটা", "নিজেকে"
}

# False positive allow-list words
ALLOW_LIST_WORDS = {
    "song", "lyrics", "music", "sing", "singing", "album", "artist", "track", "gan", "gaan", "sangeet"
}

def safety_guard_skill(user_input: str) -> dict:
    """
    Scans the user_input for crisis keywords/phrases using a tiered approach.
    - Tier 1: Immediate trigger on high-severity words (e.g. self-harm, suicide).
    - Tier 2: Trigger on lower-severity terms if 2+ are present OR combined with personal pronouns/feeling indicators.
    - Tier 3: Bypassed if allow-listed words (song lyrics references) are detected without high-severity direct intent.
    Returns a dictionary indicating if crisis was detected and localized helpline details.
    """
    if not user_input or not isinstance(user_input, str):
        return {"crisis_detected": False, "message": "No input provided.", "hotlines": {}}
        
    clean_input = user_input.lower().strip()
    
    # 1. Check for false positive context
    # If user mentions song/lyrics and DOES NOT explicitly say "kill myself", "want to die", or "morte chai", bypass.
    has_allow_word = any(word in clean_input for word in ALLOW_LIST_WORDS)
    has_critical_intent = any(intent in clean_input for intent in ["kill myself", "want to die", "morte chai", "suicide korbo", "attahatya", "আত্মহত্যা"])
    
    if has_allow_word and not has_critical_intent:
        return {"crisis_detected": False, "message": "Song lyrics/music query detected and allowed.", "hotlines": {}}
    
    # 2. Check High-Severity Terms
    triggered_high = []
    for term in HIGH_SEVERITY_TERMS:
        if re.search(r'\b' + re.escape(term) + r'\b', clean_input) or (not term.isalnum() and term in clean_input):
            triggered_high.append(term)
            
    if triggered_high:
        return trigger_crisis_response("High severity term matched: " + ", ".join(triggered_high))
        
    # 3. Check Lower-Severity Terms & Context
    matched_low = []
    for term in LOWER_SEVERITY_TERMS:
        if re.search(r'\b' + re.escape(term) + r'\b', clean_input) or (not term.isalnum() and term in clean_input):
            matched_low.append(term)
            
    matched_personal = []
    for marker in PERSONAL_CONTEXT_MARKERS:
        if marker in clean_input:
            matched_personal.append(marker)
            
    # Trigger if:
    # A) 2 or more lower severity terms matched
    # B) 1 or more lower severity terms matched AND personal context markers matched
    if len(matched_low) >= 2:
        return trigger_crisis_response(f"Multiple lower severity terms matched: {matched_low}")
    elif len(matched_low) >= 1 and len(matched_personal) >= 1:
        return trigger_crisis_response(f"Hopelessness term {matched_low} matched with personal context {matched_personal}")
        
    return {"crisis_detected": False, "message": "No crisis indicators detected.", "hotlines": {}}

def trigger_crisis_response(reason: str) -> dict:
    HELPLINES = {
        "Bangladesh Primary (Kaan Pete Roi)": {
            "Main number": "09612-119911 (or +880 9612-119911) — 3 PM to 3 AM daily",
            "GP": "01779-554391 / 01779-554392",
            "Airtel": "01688-709965 / 01688-709966",
            "Banglalink": "01985-275286",
            "Website": "kaanpeteroi.org",
            "Facebook": "Kaan Pete Roi"
        },
        "Bangladesh Emergency Options": {
            "National Emergency": "999 (Police / Immediate Crisis)",
            "Mindspace / Vent": "09678-678-778 (Psychological crisis support, evening hours)",
            "Talk Hope Bangladesh": "+880 9638-881-888",
            "Child Helpline": "1098"
        },
        "Global / International Support": {
            "Directory": "International helplines via findahelpline.com",
            "US / Canada / Standard": "988 Suicide & Crisis Lifeline"
        }
    }
    
    bilingual_message = (
        "আপনি একা নন। এখনই সাহায্য নিন।\n"
        "You are not alone. Please reach out for help right now.\n\n"
        "📞 Kaan Pete Roi (Emotional Support & Suicide Prevention): 09612-119911\n"
        "📞 National Emergency: 999\n\n"
        "Someone is ready to listen confidentially. Take one step — call now.\n"
        "Note: This is a temporary pause in the session for your safety. We care about you."
    )
    
    return {
        "crisis_detected": True,
        "reason": reason,
        "message": bilingual_message,
        "hotlines": HELPLINES
    }

# Wrap as a Google ADK FunctionTool
try:
    safety_guard_tool = FunctionTool(func=safety_guard_skill)
    print("✅ Safety Guard tool successfully registered with ADK FunctionTool.")
except Exception as e:
    print(f"⚠️ Could not wrap safety_guard_skill in FunctionTool (ADK environment missing): {e}")


✅ Safety Guard tool successfully registered with ADK FunctionTool.


In [10]:
# Automated Verification Tests for Safety Guard Skill
print("--- Running Safety Guard Verification Tests ---")

# Test case 1: Clean input
res1 = safety_guard_skill("I feel a bit tired today but overall okay.")
assert res1["crisis_detected"] is False, "Failed clean input test"
print("✅ Test Case 1: Clean input - PASSED")

# Test case 2: Clear crisis input (English)
res2 = safety_guard_skill("I want to end my life, I cannot do this anymore.")
assert res2["crisis_detected"] is True, "Failed crisis input test (English)"
assert "Bangladesh Primary (Kaan Pete Roi)" in res2["hotlines"], "Failed hotlines include test"
print("✅ Test Case 2: Crisis input (English) - PASSED")

# Test case 3: Crisis input (Transliterated Bengali)
res3 = safety_guard_skill("ami morbo, sob sesh")
assert res3["crisis_detected"] is True, "Failed crisis input test (Bengali)"
print("✅ Test Case 3: Crisis input (Bengali) - PASSED")

# Test case 4: Lower-severity term ("worthless") combined with pronoun ("I feel") -> triggers crisis
res4 = safety_guard_skill("Sometimes I feel worthless, like a burden to my family.")
assert res4["crisis_detected"] is True, "Failed lower-severity + context test"
print("✅ Test Case 4: Lower-severity + Context - PASSED")

# Test case 5: Single lower-severity term ("burden") without context -> does not trigger
res5 = safety_guard_skill("Carrying this heavy bag is a real burden.")
assert res5["crisis_detected"] is False, "Failed single lower-severity no context test"
print("✅ Test Case 5: Single lower-severity without context - PASSED")

# Test case 6: Multiple lower-severity terms ("hopeless", "trapped") -> triggers crisis
res6 = safety_guard_skill("Everything is hopeless and I feel trapped in this situation.")
assert res6["crisis_detected"] is True, "Failed multiple lower-severity test"
print("✅ Test Case 6: Multiple lower-severity terms - PASSED")

# Test case 7: Allow-list false positive mitigation
res7 = safety_guard_skill("Can you recommend a song about suicide or feeling hopeless?")
assert res7["crisis_detected"] is False, "Failed song lyrics bypass test"
print("✅ Test Case 7: Allow-list false positive mitigation - PASSED")

print("\n🎉 All Safety Guard verification tests passed successfully!")


--- Running Safety Guard Verification Tests ---
✅ Test Case 1: Clean input - PASSED
✅ Test Case 2: Crisis input (English) - PASSED
✅ Test Case 3: Crisis input (Bengali) - PASSED
✅ Test Case 4: Lower-severity + Context - PASSED
✅ Test Case 5: Single lower-severity without context - PASSED
✅ Test Case 6: Multiple lower-severity terms - PASSED
✅ Test Case 7: Allow-list false positive mitigation - PASSED

🎉 All Safety Guard verification tests passed successfully!


## 🧠 8. Natural Language Diagnostic Agent (Phase 2.3)
We implement the **Diagnostic Agent** (`LlmAgent`) and its associated custom tool `set_mood_coordinates`. 
The Diagnostic Agent listens to natural language mood descriptions (Option A/B) and translates them into Valence-Arousal coordinates.


In [11]:
    """
    session_state["current_mood"] = category
    session_state["current_coords"] = (round(float(valence), 3), round(float(arousal), 3))
    return f"Successfully registered mood: '{category}' at Valence={valence}, Arousal={arousal}."

# Wrap as a FunctionTool
set_mood_coordinates_tool = FunctionTool(func=set_mood_coordinates)

DIAGNOSTIC_INSTRUCTIONS = """
You are an empathetic, professional clinical music therapist's assistant (Diagnostic Agent).
Your primary goal is to analyze the user's natural language mood description and extract their current Valence-Arousal coordinates.

Here is the predefined 12-box named emotional model coordinates reference:
1. "Afraid / Anxious": V=-0.6, A=0.8
2. "Angry / Tense": V=-0.7, A=0.6
3. "Distressed / Annoyed": V=-0.8, A=0.2
4. "Sad / Gloomy": V=-0.7, A=-0.4
5. "Depressed / Miserable": V=-0.6, A=-0.7
6. "Bored / Tired": V=-0.3, A=-0.8
7. "Sleepy / Sluggish": V=0.0, A=-0.9
8. "Calm / Relaxed": V=0.7, A=-0.7
9. "Content / At Ease": V=0.8, A=-0.3
10. "Happy / Pleased": V=0.9, A=0.1
11. "Joyous / Excited": V=0.7, A=0.6
12. "Surprised / Alert": V=0.0, A=0.8

CRITICAL INSTRUCTION: You must make every effort to map the user's input to one of the 12 predefined emotional categories above (by number or name) (Russell's circumplex model). Only use Fallback Mode (Option A / 'Custom') if the user describes a state that is completely unrepresented by the 12 categories or is highly contradictory. Prioritize matching the closest semantic category to leverage clinical music alignment.

Guidance:
1. Read the user's statement.
2. Primary Mode (Option B - Strongly Preferred): Identify the closest matching box from the 12 named categories above. If the user's statement corresponds directly to one of these states, call the `set_mood_coordinates` tool with that category name, valence, and arousal values.
3. Fallback Mode (Option A): If the user's description is highly complex, mixed, or unique and does not align well with a single named category, estimate custom valence and arousal coordinates (floats in the range -1.0 to 1.0) and call the `set_mood_coordinates` tool with category="Custom" and your estimated valence and arousal.
4. Be warm and comforting. Explain what mood/coordinates you detected and that the session coordinator will use this to generate their therapeutic trajectory.
"""

try:
    DiagnosticAgent = Agent(
        name="DiagnosticAgent",
        model=GEMINI_MODEL,
        instruction=DIAGNOSTIC_INSTRUCTIONS,
        tools=[set_mood_coordinates_tool]
    )
    print("✅ DiagnosticAgent successfully initialized.")
except Exception as e:
    print(f"⚠️ Failed to initialize DiagnosticAgent: {e}")


✅ DiagnosticAgent successfully initialized.


## 🎛️ 9. Therapist Session Coordinator Agent (Phase 2.4)
We implement the **Coordinator Agent** (`LlmAgent`) and its customized ADK FunctionTools for onboarding, user profile checking, and goal selection.


In [12]:
def check_user_profile() -> str:
    """
    Checks if a user profile exists in LTM. Loads preference and baseline data into session state.
    """
    profile = load_user_profile()
    if profile:
        session_state["baseline_loaded"] = True
        session_state["preferred_genre"] = profile["baseline"]["preferred_genre"]
        session_state["gad2_score"] = profile["baseline"]["gad2_score"]
        return f"Profile found: Preferred Genre = {session_state['preferred_genre']}, GAD-2 Score = {session_state['gad2_score']}."
    else:
        session_state["baseline_loaded"] = False
        return "No user profile found. Onboarding required."

def run_onboarding(preferred_genre: str, gad2_score: int) -> str:
    """
    Registers a new user in LTM with preferred genre and GAD-2 score.
    """
    try:
        create_user_profile(gad2_score=int(gad2_score), preferred_genre=preferred_genre)
        session_state["baseline_loaded"] = True
        session_state["preferred_genre"] = preferred_genre
        session_state["gad2_score"] = int(gad2_score)
        return "Onboarding completed successfully. Profile created."
    except Exception as e:
        return f"Error completing onboarding: {e}"

def select_session_goal(goal_type: str) -> str:
    """
    Sets the target coordinates and generates the deterministic ISO trajectory.
    """
    goal_clean = goal_type.lower().strip()
    if "calm" in goal_clean:
        session_state["goal_type"] = "calm"
        session_state["target_coords"] = (0.75, -0.75)
    elif "focus" in goal_clean or "study" in goal_clean:
        session_state["goal_type"] = "focus"
        session_state["target_coords"] = (0.50, -0.20)
    else:
        return "Error: Invalid goal. Please choose 'calm' or 'focus'."

    if not session_state["current_coords"]:
        return "Error: Current mood coordinates are not set. Run mood check-in first."
        
    trajectory = generate_trajectory(
        start_coords=session_state["current_coords"],
        target_coords=session_state["target_coords"],
        steps=3
    )
    session_state["trajectory"] = trajectory
    return f"Session goal set to {session_state['goal_type']}. Path generated: {trajectory}"

# Wrap as FunctionTools
check_profile_tool = FunctionTool(func=check_user_profile)
onboarding_tool = FunctionTool(func=run_onboarding)
select_goal_tool = FunctionTool(func=select_session_goal)

COORDINATOR_INSTRUCTIONS = """
You are the Therapist Session Coordinator Agent.
Your task is to manage the mood tracking and goal settings for the music therapy session, and then guide them track-by-track.

CRITICAL INSTRUCTIONS:
- You must call your tools immediately when the corresponding workflow step is reached, without chatting first or asking for user permission.
- Do NOT call `feedback_tool` during the initial check-in (step 1). The `feedback_tool` is ONLY for track feedback (step 4) after a track has played.

Workflow:
1. Ask the user how they are currently feeling. When asking, print the numbered list of 12 emotions clearly to let the user select by number (1-12), type the emotion name, or describe their mood in free-text:
   1. Afraid / Anxious
   2. Angry / Tense
   3. Distressed / Annoyed
   4. Sad / Gloomy
   5. Depressed / Miserable
   6. Bored / Tired
   7. Sleepy / Sluggish
   8. Calm / Relaxed
   9. Content / At Ease
   10. Happy / Pleased
   11. Joyous / Excited
   12. Surprised / Alert
   Route the user's response to `DiagnosticAgent` to get their starting coordinates.
2. Ask the user for their therapy goal (either 'calm' or 'focus' / 'study') and call `select_goal_tool` with the goal.
3. After the goal is set, call `get_track_tool` to get the first track recommendation and display it (Title, Artist, Link) to the user.
4. Once the user has listened, ask them how they feel. They can enter a mood index (1-12), emotion name, or free-text description.
   - If they enter a number (1-12) or exact/partial name, call `feedback_tool` directly with their selection.
   - If they enter a free-text description, route it to `DiagnosticAgent` first to resolve the category name and coordinates, and then call `feedback_tool` with the resolved category name.
5. Call `get_track_tool` to present the next track, repeating feedback check-ins.
6. After the final track, call `conclude_tool` to save progress to LTM and print the concluding exercises.

Be warm, empathetic, and guide the user through these steps sequentially in conversation. Always list the 12 emotions clearly at the check-in step (step 1) so they have the choice.
"""

try:
    CoordinatorAgent = Agent(
        name="CoordinatorAgent",
        model=GEMINI_MODEL,
        instruction=COORDINATOR_INSTRUCTIONS,
        tools=[select_goal_tool, get_track_tool, feedback_tool, conclude_tool],
        sub_agents=[DiagnosticAgent]
    )
    print("✅ CoordinatorAgent successfully initialized.")
except Exception as e:
    print(f"⚠️ Failed to initialize CoordinatorAgent: {e}")


✅ CoordinatorAgent successfully initialized.


## 🧪 10. Verification of Phase 2.3 & Phase 2.4 Agents
We run automated checks and stream queries using the `Runner` to verify that our agents interact with their tools correctly.


In [13]:
import asyncio
from google.genai import types

# Apply nest_asyncio to support running nested event loops in Jupyter
try:
    import nest_asyncio
    nest_asyncio.apply()
    print("✅ nest_asyncio applied successfully.")
except Exception as e:
    print(f"⚠️ Could not apply nest_asyncio: {e}")

async def verify_agents():
    # 1. Reset state and load/create session
    global session_state
    session_state = {
        "session_id": "session_verify_123",
        "user_id": "user_01",
        "baseline_loaded": False,
        "preferred_genre": None,
        "gad2_score": None,
        "current_mood": None,
        "current_coords": None,
        "target_coords": None,
        "goal_type": None,
        "trajectory": None,
        "playlist": [],
        "played_ids": set(),
        "current_step": 0
    }
    
    session_service = InMemorySessionService()
    
    # Create runner for DiagnosticAgent with app_name="agents" to avoid mismatch errors
    diag_runner = Runner(
        agent=DiagnosticAgent,
        session_service=session_service,
        app_name="agents"
    )
    session_diag = await session_service.create_session(app_name="agents", user_id="user_01")
    
    print("\n--- 1. Testing Diagnostic Agent (Option B mapping) ---")
    msg_diag = types.Content(parts=[types.Part(text="I'm feeling really angry and tense right now.")])
    
    async for event in diag_runner.run_async(
        user_id="user_01",
        session_id=session_diag.id,
        new_message=msg_diag
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text, end="")
    print(f"\nResulting Mood State: {session_state['current_mood']} at {session_state['current_coords']}")
    assert session_state["current_mood"] == "Angry / Tense", "DiagnosticAgent failed to map to 'Angry / Tense'"
    
    print("\n--- 2. Testing Diagnostic Agent (Option A custom coordinates) ---")
    msg_diag_custom = types.Content(parts=[types.Part(text="I feel somewhat calm but also slightly excited, like a mixed calm-energized feeling.")])
    session_diag_custom = await session_service.create_session(app_name="agents", user_id="user_01")
    
    async for event in diag_runner.run_async(
        user_id="user_01",
        session_id=session_diag_custom.id,
        new_message=msg_diag_custom
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text, end="")
    print(f"\nResulting Custom Mood Coordinates: {session_state['current_coords']}")
    assert session_state["current_coords"] is not None, "Failed to parse coordinates for custom state"

    # Create runner for CoordinatorAgent with app_name="agents" to avoid mismatch errors
    coord_runner = Runner(
        agent=CoordinatorAgent,
        session_service=session_service,
        app_name="agents"
    )
    session_coord = await session_service.create_session(app_name="agents", user_id="user_01")
    
    print("\n--- 3. Testing Coordinator Agent (Profile Checking & Goal) ---")
    # Onboard the user by feeding onboarding info to Coordinator
    msg_coord = types.Content(parts=[types.Part(text="Hello! Please check my profile. My mood check-in is complete and coordinates are set. Please select my session goal as calm.")])
    
    async for event in coord_runner.run_async(
        user_id="user_01",
        session_id=session_coord.id,
        new_message=msg_coord
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text, end="")
                    
    print(f"\nCoordinator Session Goal: {session_state['goal_type']}")
    print(f"Generated Trajectory Path: {session_state['trajectory']}")
    assert session_state["goal_type"] == "calm", "CoordinatorAgent failed to set session goal"
    assert session_state["trajectory"] is not None, "CoordinatorAgent failed to generate trajectory"
    
    print("\n🎉 All Agent Verification Tests Passed Successfully!")

# Run verification function
import os
if os.environ.get("GEMINI_API_KEY"):
    import asyncio
    loop = asyncio.get_event_loop()
    if loop.is_running():
        task = loop.create_task(verify_agents())
    else:
        loop.run_until_complete(verify_agents())
else:
    print("⚠️ GEMINI_API_KEY missing. Skipping live verification tests.")


✅ nest_asyncio applied successfully.



--- 1. Testing Diagnostic Agent (Option B mapping) ---


## 🔄 11. Phase 3: Active Feedback & Recalibration - Task 1: Unified Feedback UI
We implement the core subtasks of Phase 3 Task 1 (Unified Feedback UI):
- **1.1 12-Box Visual & Selection Menu**: A function to render the 12 emotions in an interactive HTML grid (using `IPython.display.HTML`) and a text-based fallback.
- **1.2 Previous State Tracking**: Visual indication of the user's previously recorded state (using cyan border highlighting in HTML and `[PREV]` prefixes in text).
- **1.3 Robust Input Validation**: Validates user inputs (accepts 1-12 or mood name) and handles partial/case-insensitive matches.

In [14]:
# Predefined emotion list from Russell's circumplex model
EMOTION_LIST = [
    {"name": "Afraid / Anxious", "coords": (-0.6, 0.8), "color": "#E74C3C"},
    {"name": "Angry / Tense", "coords": (-0.7, 0.6), "color": "#C0392B"},
    {"name": "Distressed / Annoyed", "coords": (-0.8, 0.2), "color": "#D35400"},
    {"name": "Sad / Gloomy", "coords": (-0.7, -0.4), "color": "#7F8C8D"},
    {"name": "Depressed / Miserable", "coords": (-0.6, -0.7), "color": "#34495E"},
    {"name": "Bored / Tired", "coords": (-0.3, -0.8), "color": "#95A5A6"},
    {"name": "Sleepy / Sluggish", "coords": (0.0, -0.9), "color": "#BDC3C7"},
    {"name": "Calm / Relaxed", "coords": (0.7, -0.7), "color": "#2ECC71"},
    {"name": "Content / At Ease", "coords": (0.8, -0.3), "color": "#27AE60"},
    {"name": "Happy / Pleased", "coords": (0.9, 0.1), "color": "#F1C40F"},
    {"name": "Joyous / Excited", "coords": (0.7, 0.6), "color": "#E67E22"},
    {"name": "Surprised / Alert", "coords": (0.0, 0.8), "color": "#9B59B6"}
]

def render_html_emotion_grid(previous_mood=None):
    """
    Renders the 12-box named emotional model in a beautiful, styled grid in Jupyter.
    If previous_mood is specified, it highlights the matching card with a special cyan border.
    """
    from IPython.display import display, HTML
    
    html_code = """
    <style>
        .circumplex-container {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
            gap: 12px;
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
            max-width: 800px;
            margin: 20px auto;
            background: #1a1a2e;
            padding: 20px;
            border-radius: 16px;
            box-shadow: 0 8px 32px rgba(0,0,0,0.3);
        }
        .emotion-box {
            padding: 15px;
            border-radius: 12px;
            text-align: center;
            color: #ffffff;
            font-weight: 600;
            font-size: 0.9em;
            transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
            border: 2px solid transparent;
            box-shadow: 0 4px 15px rgba(0,0,0,0.15);
        }
        .emotion-coords {
            display: block;
            font-size: 0.75em;
            font-weight: 400;
            opacity: 0.85;
            margin-top: 4px;
        }
        .emotion-index {
            display: inline-block;
            background: rgba(255,255,255,0.25);
            border-radius: 50%;
            width: 18px;
            height: 18px;
            line-height: 18px;
            font-size: 0.75em;
            margin-right: 6px;
            text-align: center;
        }
        .previous-state {
            border: 3px solid #00E6FF !important;
            animation: pulse-border-feedback 2s infinite;
            box-shadow: 0 0 20px rgba(0, 230, 255, 0.6);
            transform: scale(1.03);
        }
        @keyframes pulse-border-feedback {
            0% { box-shadow: 0 0 0 0 rgba(0, 230, 255, 0.7); }
            70% { box-shadow: 0 0 0 10px rgba(0, 230, 255, 0); }
            100% { box-shadow: 0 0 0 0 rgba(0, 230, 255, 0); }
        }
    </style>
    <div style="text-align: center; font-family: sans-serif; color: #ffffff; margin-top: 15px;">
        <h3 style="margin-bottom: 5px; background: linear-gradient(45deg, #FF6B6B, #4A90E2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Select Your Current Emotional State</h3>
        <p style="font-size: 0.9em; opacity: 0.8; margin-top: 0;">Highlighting previous state in cyan.</p>
    </div>
    <div class="circumplex-container">
    """
    
    for idx, emo in enumerate(EMOTION_LIST, 1):
        is_prev = (previous_mood is not None) and (emo["name"].lower() == previous_mood.lower())
        prev_class = " previous-state" if is_prev else ""
        prev_label = " 🌟 (Previous)" if is_prev else ""
        
        html_code += f"""
        <div class="emotion-box{prev_class}" style="background: {emo['color']};">
            <div><span class="emotion-index">{idx}</span>{emo['name']}{prev_label}</div>
            <span class="emotion-coords">V: {emo['coords'][0]:.1f}, A: {emo['coords'][1]:.1f}</span>
        </div>
        """
        
    html_code += "</div>"
    display(HTML(html_code))

def render_text_emotion_grid(previous_mood=None):
    """
    Fallback text renderer for terminal/console environments.
    """
    print("="*80)
    print("                       SELECT YOUR CURRENT EMOTIONAL STATE                      ")
    print("="*80)
    for idx in range(0, len(EMOTION_LIST), 2):
        emo1 = EMOTION_LIST[idx]
        emo2 = EMOTION_LIST[idx+1] if idx+1 < len(EMOTION_LIST) else None
        
        is_prev1 = previous_mood and emo1["name"].lower() == previous_mood.lower()
        star1 = "[PREV] " if is_prev1 else ""
        label1 = f"{idx+1:2d}. {star1}{emo1['name']} (V:{emo1['coords'][0]:.1f}, A:{emo1['coords'][1]:.1f})"
        
        if emo2:
            is_prev2 = previous_mood and emo2["name"].lower() == previous_mood.lower()
            star2 = "[PREV] " if is_prev2 else ""
            label2 = f"{idx+2:2d}. {star2}{emo2['name']} (V:{emo2['coords'][0]:.1f}, A:{emo2['coords'][1]:.1f})"
            print(f"{label1:<40} | {label2}")
        else:
            print(f"{label1}")
    print("="*80)

def get_user_feedback(previous_mood=None, test_input=None):
    """
    Displays the 12-box mood selector and performs robust validation.
    Accepts either:
    - Index numbers from 1 to 12
    - Partial or exact mood names (case-insensitive)
    """
    try:
        # Check if IPython is running and HTML is supported
        from IPython import get_ipython
        ip = get_ipython()
        if ip is not None:
            render_html_emotion_grid(previous_mood)
        else:
            render_text_emotion_grid(previous_mood)
    except Exception:
        render_text_emotion_grid(previous_mood)
        
    while True:
        if test_input is not None:
            user_sel = str(test_input).strip()
            print(f"User selection (mocked): '{user_sel}'")
        else:
            user_sel = input("Enter option number (1-12) or mood name: ").strip()
            
        if not user_sel:
            print("❌ Input cannot be empty. Please try again.")
            if test_input is not None:
                raise ValueError("Empty input received in automated test.")
            continue
            
        # Try matching by number index
        if user_sel.isdigit():
            val = int(user_sel)
            if 1 <= val <= 12:
                selected = EMOTION_LIST[val - 1]
                print(f"✅ Selection validated: '{selected['name']}' at {selected['coords']}")
                return selected
            else:
                print("❌ Invalid option index! Please enter a number between 1 and 12.")
        else:
            # Try matching by name (exact first, then partial case-insensitive)
            clean_sel = user_sel.lower()
            matches = [e for e in EMOTION_LIST if e["name"].lower() == clean_sel]
            if not matches:
                # Try partial match
                matches = [e for e in EMOTION_LIST if clean_sel in e["name"].lower()]
                
            if len(matches) == 1:
                selected = matches[0]
                print(f"✅ Selection validated: '{selected['name']}' at {selected['coords']}")
                return selected
            elif len(matches) > 1:
                print(f"❌ Ambiguous input. Multiple matches found: {', '.join([m['name'] for m in matches])}. Please select again.")
            else:
                print("❌ Mood name not recognized. Please try again or enter the corresponding number (1-12).")
                
        if test_input is not None:
            # Break out of loop if in automated testing and invalid input was encountered
            raise ValueError(f"Invalid selection: '{test_input}'")


In [15]:
# Verification Tests for Phase 3 Task 1
print("--- Running Unified Feedback UI Verification Tests ---")

# 1. Test numeric input validation
res_num = get_user_feedback(previous_mood="Angry / Tense", test_input="8")
assert res_num["name"] == "Calm / Relaxed", "Numeric validation failed"
print("✅ Test 1: Numeric Input Validation (Option 8 -> Calm / Relaxed) Passed.")

# 2. Test exact name matching
res_exact = get_user_feedback(previous_mood=None, test_input="Angry / Tense")
assert res_exact["name"] == "Angry / Tense", "Exact name validation failed"
print("✅ Test 2: Exact Name Validation (Angry / Tense) Passed.")

# 3. Test partial case-insensitive matching
res_partial = get_user_feedback(previous_mood=None, test_input="anxious")
assert res_partial["name"] == "Afraid / Anxious", "Partial matching failed"
print("✅ Test 3: Partial Name Validation (anxious -> Afraid / Anxious) Passed.")

# 4. Test invalid input handling (should raise ValueError when test_input is set)
try:
    get_user_feedback(previous_mood=None, test_input="invalid mood label")
    assert False, "Failed to raise error on invalid input"
except ValueError:
    print("✅ Test 4: Invalid Input Rejection Passed.")

# 5. Test previous mood highlight trigger (Visual test representation)
print("\nVisual representation of grid with highlight active:")
render_text_emotion_grid(previous_mood="Calm / Relaxed")

print("\n🎉 All Phase 3 Task 1 Verification Tests Passed Successfully!")


--- Running Unified Feedback UI Verification Tests ---


User selection (mocked): '8'
✅ Selection validated: 'Calm / Relaxed' at (0.7, -0.7)
✅ Test 1: Numeric Input Validation (Option 8 -> Calm / Relaxed) Passed.


User selection (mocked): 'Angry / Tense'
✅ Selection validated: 'Angry / Tense' at (-0.7, 0.6)
✅ Test 2: Exact Name Validation (Angry / Tense) Passed.


User selection (mocked): 'anxious'
✅ Selection validated: 'Afraid / Anxious' at (-0.6, 0.8)
✅ Test 3: Partial Name Validation (anxious -> Afraid / Anxious) Passed.


User selection (mocked): 'invalid mood label'
❌ Mood name not recognized. Please try again or enter the corresponding number (1-12).
✅ Test 4: Invalid Input Rejection Passed.

Visual representation of grid with highlight active:
                       SELECT YOUR CURRENT EMOTIONAL STATE                      
 1. Afraid / Anxious (V:-0.6, A:0.8)     |  2. Angry / Tense (V:-0.7, A:0.6)
 3. Distressed / Annoyed (V:-0.8, A:0.2) |  4. Sad / Gloomy (V:-0.7, A:-0.4)
 5. Depressed / Miserable (V:-0.6, A:-0.7) |  6. Bored / Tired (V:-0.3, A:-0.8)
 7. Sleepy / Sluggish (V:0.0, A:-0.9)    |  8. [PREV] Calm / Relaxed (V:0.7, A:-0.7)
 9. Content / At Ease (V:0.8, A:-0.3)    | 10. Happy / Pleased (V:0.9, A:0.1)
11. Joyous / Excited (V:0.7, A:0.6)      | 12. Surprised / Alert (V:0.0, A:0.8)

🎉 All Phase 3 Task 1 Verification Tests Passed Successfully!


## 🔄 12. Phase 3: Active Feedback & Recalibration - Task 2: Transition Logic & Classification
We implement the subtasks of Phase 3 Task 2:
- **2.1 Coordinate Delta Calculation**: Calculate the Valence and Arousal coordinate shifts between the user's previous mood state and the current mood check-in.
- **2.2 State Transition Categorization**: Clinical rules to classify transitions as `'calmer'`, `'no_change'`, or `'worse'`.

In [16]:
def classify_emotional_transition(prev_coords, new_coords, prev_name=None, new_name=None, threshold=0.05):
    """
    Compares the coordinates of the new state (V_i, A_i) against the previous state (V_i-1, A_i-1).
    Calculates the shift delta (Task 2.1) and labels the transition (Task 2.2) as:
    - 'calmer' (Valence increased and/or Arousal decreased)
    - 'no_change' (Same box or negligible shift)
    - 'worse' (Valence decreased and/or Arousal increased)
    """
    v_prev, a_prev = prev_coords
    v_new, a_new = new_coords
    
    # 2.1 Coordinate Delta Calculation
    v_delta = round(v_new - v_prev, 3)
    a_delta = round(a_new - a_prev, 3)
    
    # 2.2 State Transition Categorization
    # Check exact name match first for no_change
    if prev_name and new_name and prev_name.lower().strip() == new_name.lower().strip():
        return "no_change"
        
    # Calculate Euclidean distance
    dist = np.sqrt(v_delta**2 + a_delta**2)
    if dist < threshold:
        return "no_change"
        
    # Clinically, worse means increased agitation (arousal goes up) or mood drop (valence goes down)
    # Let's flag worse if arousal goes up (> 0.05) and valence does not significantly compensate,
    # or if valence drops (< -0.05) and arousal does not significantly compensate.
    if a_delta > 0.05 and v_delta <= 0.05:
        return "worse"
    if v_delta < -0.05 and a_delta >= -0.05:
        return "worse"
        
    # Clinically, calmer/better means decreased arousal (<= -0.05) or increased valence (>= 0.05)
    if a_delta <= -0.05 or v_delta >= 0.05:
        return "calmer"
        
    return "no_change"


In [17]:
# Verification Tests for Phase 3 Task 2
print("--- Running Transition Logic Verification Tests ---")

# 1. Test Calmer transition (Arousal decreased)
t1 = classify_emotional_transition(prev_coords=(-0.7, 0.6), new_coords=(-0.7, 0.2))  # Angry/Tense to Distressed (reduced arousal)
assert t1 == "calmer", f"Failed calmer test: expected calmer, got {t1}"
print("✅ Test 1: Calmer transition (arousal decrease) detected.")

# 2. Test Calmer transition (Valence increased)
t2 = classify_emotional_transition(prev_coords=(-0.7, -0.4), new_coords=(0.7, -0.7))  # Sad/Gloomy to Calm/Relaxed
assert t2 == "calmer", f"Failed calmer test: expected calmer, got {t2}"
print("✅ Test 2: Calmer transition (valence increase) detected.")

# 3. Test No Change transition (identical coords & names)
t3 = classify_emotional_transition(prev_coords=(0.7, -0.7), new_coords=(0.7, -0.7), prev_name="Calm / Relaxed", new_name="Calm / Relaxed")
assert t3 == "no_change", f"Failed no_change test: expected no_change, got {t3}"
print("✅ Test 3: No Change transition (identical states) detected.")

# 4. Test No Change transition (within negligible threshold)
t4 = classify_emotional_transition(prev_coords=(0.7, -0.7), new_coords=(0.71, -0.71))  # shift dist = 0.014
assert t4 == "no_change", f"Failed no_change test: expected no_change, got {t4}"
print("✅ Test 4: No Change transition (negligible shift) detected.")

# 5. Test Worse transition (Arousal increased)
t5 = classify_emotional_transition(prev_coords=(-0.7, -0.4), new_coords=(-0.7, 0.6))  # Sad to Angry/Tense (arousal spike)
assert t5 == "worse", f"Failed worse test: expected worse, got {t5}"
print("✅ Test 5: Worse transition (arousal increase) detected.")

# 6. Test Worse transition (Valence decreased)
t6 = classify_emotional_transition(prev_coords=(0.7, -0.7), new_coords=(-0.6, -0.7))  # Calm to Depressed (valence crash)
assert t6 == "worse", f"Failed worse test: expected worse, got {t6}"
print("✅ Test 6: Worse transition (valence decrease) detected.")

print("\n🎉 All Phase 3 Task 2 Verification Tests Passed Successfully!")


--- Running Transition Logic Verification Tests ---
✅ Test 1: Calmer transition (arousal decrease) detected.
✅ Test 2: Calmer transition (valence increase) detected.
✅ Test 3: No Change transition (identical states) detected.
✅ Test 4: No Change transition (negligible shift) detected.
✅ Test 5: Worse transition (arousal increase) detected.
✅ Test 6: Worse transition (valence decrease) detected.

🎉 All Phase 3 Task 2 Verification Tests Passed Successfully!


## 🔄 13. Phase 3: Active Feedback & Recalibration - Task 3: Recalibration Algorithms
We implement the subtasks of Phase 3 Task 3:
- **3.1 Calmer Path**: Continue planned trajectory.
- **3.2 No-Change Recalibration Engine**: Dynamically adjust remaining trajectory to lower target arousal ($A_{target} - 0.15$), avoid stale genre, and re-query database with preferred genre boost.
- **3.3 Worse State Rescue Interception**: Clear session queue, load pre-designated deep-calm rescue track (instrumental flute/sitar), queue it immediately, and raise crisis override flags.

In [18]:
def recalibrate_session_trajectory(session_state, transition_status, db_df):
    """
    Recalibrates the session trajectory and playlist based on the transition_status.
    
    - 'calmer' (3.1): Continue planned trajectory. Advance current_step.
    - 'no_change' (3.2): Reduce target arousal by 0.15, filter out stale genre, recalculate trajectory and playlist.
    - 'worse' (3.3): Immediately halt playlist, queue deep-calm rescue track, and transition to session end.
    
    Returns a status message explaining the recalibration action taken.
    """
    current_step = session_state.get("current_step", 0)
    playlist = session_state.get("playlist", [])
    
    if transition_status == "calmer":
        session_state["current_step"] = current_step + 1
        return "Proceeding along the planned therapeutic trajectory as the user is responding positively (Calmer)."
        
    elif transition_status == "no_change":
        # 1. Get current position (coordinates of the track just played)
        last_track = playlist[current_step] if current_step < len(playlist) else None
        last_coords = (last_track["valence"], last_track["arousal"]) if last_track else session_state["current_coords"]
        
        # 2. Adjust target coordinate: reduce target arousal by 0.15 (clip to -1.0)
        target_v, target_a = session_state["target_coords"]
        new_target_a = max(-1.0, target_a - 0.15)
        new_target_coords = (target_v, new_target_a)
        
        # 3. Calculate remaining steps
        total_steps = len(playlist)
        remaining_steps = total_steps - (current_step + 1)
        
        if remaining_steps <= 0:
            session_state["current_step"] = current_step + 1
            return "No Change detected, but the session is at its final step. Proceeding to conclude."
            
        # 4. Generate new trajectory path for the remaining steps
        new_traj = generate_trajectory(
            start_coords=last_coords,
            target_coords=new_target_coords,
            steps=remaining_steps + 1
        )
        new_traj_steps = new_traj[1:]
        
        # 5. Re-query tracks with adjustments:
        # - Filter out stale genre (the genre of the last played track)
        stale_genre = last_track["genre"] if last_track else None
        
        played_ids = set(session_state.get("played_ids", set()))
        if last_track:
            played_ids.add(last_track["track_id"])
            
        new_playlist_segment = []
        for i, coords in enumerate(new_traj_steps):
            # Exclude the stale genre in query
            filtered_db = db_df
            if stale_genre:
                filtered_db = db_df[db_df["genre"] != stale_genre]
                if filtered_db.empty:
                    filtered_db = db_df
                    
            matched_track = match_track_to_coordinate(
                valence_target=coords[0],
                arousal_target=coords[1],
                db_df=filtered_db,
                preferred_genre=session_state.get("preferred_genre"),
                played_ids=played_ids
            )
            if matched_track:
                new_playlist_segment.append(matched_track)
                played_ids.add(matched_track["track_id"])
            else:
                matched_track_fallback = match_track_to_coordinate(
                    valence_target=coords[0],
                    arousal_target=coords[1],
                    db_df=db_df,
                    preferred_genre=session_state.get("preferred_genre"),
                    played_ids=played_ids
                )
                if matched_track_fallback:
                    new_playlist_segment.append(matched_track_fallback)
                    played_ids.add(matched_track_fallback["track_id"])
        
        # 6. Update playlist in session state: keep played tracks, replace remaining ones
        updated_playlist = playlist[:current_step + 1] + new_playlist_segment
        session_state["playlist"] = updated_playlist
        session_state["played_ids"] = played_ids
        session_state["current_step"] = current_step + 1
        
        return f"No Change detected. Recalibrated remaining trajectory towards lower arousal target {new_target_coords}, avoiding stale genre '{stale_genre}'."
        
    elif transition_status == "worse":
        # Worse State Rescue Interception (3.3):
        # 1. Clear remaining playlist steps (keep up to current_step + 1)
        played_playlist = playlist[:current_step + 1]
        
        # 2. Pick a pre-designated clinical deep-calm rescue track
        rescue_track_id = "track_02"  # Flute instrumental (0.75, -0.75)
        if rescue_track_id in session_state.get("played_ids", set()):
            rescue_track_id = "track_24"  # Sitar instrumental (0.75, -0.75)
            
        rescue_rows = db_df[db_df["track_id"] == rescue_track_id]
        if not rescue_rows.empty:
            rescue_track = rescue_rows.iloc[0].to_dict()
        else:
            rescue_track = match_track_to_coordinate(
                valence_target=0.75,
                arousal_target=-0.75,
                db_df=db_df,
                preferred_genre=session_state.get("preferred_genre"),
                played_ids=set()
            )
            
        # 3. Force rescue track into playlist and update state
        session_state["playlist"] = played_playlist + [rescue_track]
        session_state["current_step"] = len(session_state["playlist"]) - 1
        session_state["crisis_flagged"] = True
        
        return f"WARNING: Negative state shift detected (Worse). Intercepting session and queueing deep-calm rescue track: '{rescue_track['title']}'."
        
    return "Invalid transition status."


In [19]:
# Verification Tests for Phase 3 Task 3
print("--- Running Recalibration Algorithms Verification Tests ---")

# Prepare a simulated session state and database
sim_db = df  # Use loaded dataframe

sim_state_calmer = {
    "current_step": 0,
    "playlist": [{"track_id": "track_11", "genre": "Bengali Band"}, {"track_id": "track_17", "genre": "Bengali Band"}, {"track_id": "track_02", "genre": "Rabindra Sangeet"}],
    "target_coords": (0.75, -0.75),
    "current_coords": (-0.7, 0.6),
    "played_ids": {"track_11"}
}

# 1. Test Calmer Path (3.1)
msg_calmer = recalibrate_session_trajectory(sim_state_calmer, "calmer", sim_db)
assert sim_state_calmer["current_step"] == 1, "Calmer path failed to advance step"
print("✅ Test 1: Calmer path advances step index correctly.")

# 2. Test No-Change Recalibration Engine (3.2)
sim_state_nochange = {
    "current_step": 0,
    "playlist": [
        {"track_id": "track_11", "title": "Trishula", "genre": "Bengali Band", "valence": -0.75, "arousal": 0.75},
        {"track_id": "track_17", "title": "Prithibita", "genre": "Bengali Band", "valence": 0.5, "arousal": 0.5},
        {"track_id": "track_02", "title": "Ami Chini", "genre": "Rabindra Sangeet", "valence": 0.75, "arousal": -0.75}
    ],
    "target_coords": (0.75, -0.75),
    "current_coords": (-0.7, 0.6),
    "preferred_genre": "Rabindra Sangeet",
    "played_ids": {"track_11"}
}
msg_nochange = recalibrate_session_trajectory(sim_state_nochange, "no_change", sim_db)
assert sim_state_nochange["current_step"] == 1, "No-change path failed to advance step"
# The new remaining queue should avoid the stale genre "Bengali Band"
remaining_tracks = sim_state_nochange["playlist"][1:]
for t in remaining_tracks:
    assert t["genre"] != "Bengali Band", f"Failed: recalibrated track {t['title']} has stale genre 'Bengali Band'"
print("✅ Test 2: No-Change Recalibration adjusts target, advances step, and avoids stale genre.")

# 3. Test Worse State Interception (3.3)
sim_state_worse = {
    "current_step": 0,
    "playlist": [
        {"track_id": "track_11", "title": "Trishula", "genre": "Bengali Band", "valence": -0.75, "arousal": 0.75},
        {"track_id": "track_17", "title": "Prithibita", "genre": "Bengali Band", "valence": 0.5, "arousal": 0.5},
        {"track_id": "track_02", "title": "Ami Chini", "genre": "Rabindra Sangeet", "valence": 0.75, "arousal": -0.75}
    ],
    "target_coords": (0.75, -0.75),
    "current_coords": (-0.7, 0.6),
    "played_ids": {"track_11"},
    "crisis_flagged": False
}
msg_worse = recalibrate_session_trajectory(sim_state_worse, "worse", sim_db)
# The remaining tracks should be wiped and replaced by only the rescue track
assert len(sim_state_worse["playlist"]) == 2, f"Failed worse queue override: playlist size {len(sim_state_worse['playlist'])}"
assert sim_state_worse["playlist"][-1]["track_id"] in ["track_02", "track_24"], "Worse path failed to queue rescue track"
assert sim_state_worse["current_step"] == 1, f"Expected current_step to be 1, got {sim_state_worse['current_step']}"
assert sim_state_worse["crisis_flagged"] is True, "Worse path failed to flag crisis state"
print("✅ Test 3: Worse Interception overrides queue, inserts rescue track, and sets safety override flags.")

print("\n🎉 All Phase 3 Task 3 Verification Tests Passed Successfully!")


--- Running Recalibration Algorithms Verification Tests ---
✅ Test 1: Calmer path advances step index correctly.
✅ Test 2: No-Change Recalibration adjusts target, advances step, and avoids stale genre.
✅ Test 3: Worse Interception overrides queue, inserts rescue track, and sets safety override flags.

🎉 All Phase 3 Task 3 Verification Tests Passed Successfully!


## 🔄 14. Phase 3: Active Feedback & Recalibration - Task 4: Session Concluder & Memory Integration
We implement the subtasks of Phase 3 Task 4:
- **4.1 Grounding Prompts**: Deliver breathing, sensory, or stabilization exercises depending on the user's final coordinates.
- **4.2 LTM Logging**: Record full session summaries (timestamp, goal, start/end coordinates, outcome status, played tracks) to local JSON memory (`user_profile.json`).
- **4.3 Baseline Profiling & Adjustments**: Dynamic baseline preferred genre updates in LTM when positive shifts are caused by different genres, and overall success rate metrics tracking.

In [20]:
def generate_grounding_exercises(final_coords):
    """
    Generates tailored grounding prompts and breathing exercises based on the ending coordinates.
    """
    v, a = final_coords
    
    # Check if final state has high arousal (needs calming) or negative valence (needs uplifting)
    if a > 0.0:
        return (
            "🧘 [Clinical Grounding Exercise: 4-7-8 Breathing]\n"
            "To help settle your active energy, let's practice the 4-7-8 breathing technique:\n"
            "1. Inhale quietly through your nose for 4 seconds.\n"
            "2. Hold your breath for a count of 7 seconds.\n"
            "3. Exhale completely through your mouth, making a whoosh sound, for 8 seconds.\n"
            "Repeat this cycle 4 times to ground your nervous system."
        )
    elif v < 0.2:
        return (
            "🌱 [Clinical Grounding Exercise: 5-4-3-2-1 Sensory Focus]\n"
            "To help ground your mind and ease feelings of somberness or isolation, let's scan your surroundings:\n"
            "- Acknowledge 5 things you can see around you.\n"
            "- Acknowledge 4 things you can touch.\n"
            "- Acknowledge 3 things you can hear.\n"
            "- Acknowledge 2 things you can smell.\n"
            "- Acknowledge 1 thing you can taste.\n"
            "Take a deep, slow breath. You are here, and you are safe."
        )
    else:
        return (
            "✨ [Clinical Grounding Exercise: Positive Stabilization]\n"
            "You have reached a serene and balanced state. Take a moment to reflect on this stability:\n"
            "1. Sit comfortably and notice the relaxed feeling in your chest and shoulders.\n"
            "2. Anchor this sensation in your memory. Whenever you feel stressed, remember this calm focus.\n"
            "We wish you a wonderful and peaceful day ahead!"
        )

def conclude_session_and_save_ltm(session_state, transition_history):
    """
    Saves the session outcome to LTM (Task 4.2), performs baseline adjustments (Task 4.3),
    and generates the concluding grounding instructions (Task 4.1).
    """
    # 1. Compile session summary
    starting_coords = session_state.get("initial_coords")
    final_coords = session_state.get("current_coords")
    
    # Outcome classification
    is_rescued = session_state.get("crisis_flagged", False)
    outcome = "rescued" if is_rescued else "successful"
    
    session_summary = {
        "session_id": session_state.get("session_id", "session_unknown"),
        "date": datetime.now().isoformat(),
        "starting_coords": list(starting_coords) if starting_coords else None,
        "final_coords": list(final_coords) if final_coords else None,
        "goal_type": session_state.get("goal_type"),
        "tracks_played": list(session_state.get("played_ids", [])),
        "transition_history": transition_history,
        "outcome": outcome,
        "completed": True
    }
    
    # 4.2 Save to LTM
    add_session_to_history(session_summary)
    
    # 4.3 Baseline Profiling & Adjustments
    profile = load_user_profile()
    if profile:
        # Extract genres that led to a positive shift
        positive_genres = [h["genre"] for h in transition_history if h.get("status") == "calmer"]
        
        if positive_genres:
            from collections import Counter
            most_successful_genre = Counter(positive_genres).most_common(1)[0][0]
            
            old_preferred = profile["baseline"].get("preferred_genre")
            if most_successful_genre != old_preferred:
                profile["baseline"]["preferred_genre"] = most_successful_genre
                profile["baseline"]["updated_at"] = datetime.now().isoformat()
                print(f"📊 [LTM Baseline Adjustment]: Updated preferred genre from '{old_preferred}' to '{most_successful_genre}' based on positive shift.")
                
        # Update metrics
        history = profile.get("session_history", [])
        total_sessions = len(history)
        success_sessions = sum(1 for s in history if s.get("outcome") == "successful")
        
        profile["baseline"]["total_sessions"] = total_sessions
        profile["baseline"]["success_rate"] = round(success_sessions / total_sessions, 2) if total_sessions > 0 else 1.0
        save_user_profile(profile)
        
    # 4.1 Tailored Grounding exercises
    grounding = generate_grounding_exercises(final_coords if final_coords else (0.0, 0.0))
    return grounding


In [21]:
# Verification Tests for Phase 3 Task 4
print("--- Running Session Concluder & LTM Verification Tests ---")

# 1. Test Grounding Prompts (4.1)
ex_calm = generate_grounding_exercises(final_coords=(0.7, -0.7))  # Calm
assert "Positive Stabilization" in ex_calm, "Failed calm grounding"

ex_high_arousal = generate_grounding_exercises(final_coords=(-0.6, 0.8))  # Anxious
assert "4-7-8 Breathing" in ex_high_arousal, "Failed high arousal grounding"

ex_sad = generate_grounding_exercises(final_coords=(-0.7, -0.4))  # Sad
assert "5-4-3-2-1 Sensory Focus" in ex_sad, "Failed low valence grounding"
print("✅ Test 1: tailored grounding exercise prompts generated correctly for all ending coordinates.")

# 2. Test LTM Logging and Baseline Updates (4.2 & 4.3)
if os.path.exists(PROFILE_PATH):
    os.remove(PROFILE_PATH)
create_user_profile(gad2_score=3, preferred_genre="Rabindra Sangeet")

sim_state_success = {
    "session_id": "session_concluder_verify",
    "initial_coords": (-0.7, 0.6),
    "current_coords": (0.7, -0.7),
    "goal_type": "calm",
    "played_ids": {"track_11", "track_02"},
    "crisis_flagged": False
}

sim_transitions = [
    {"step": 1, "genre": "Bengali Band", "status": "no_change"},
    {"step": 2, "genre": "Modern Acoustic", "status": "calmer"}
]

grounding_msg = conclude_session_and_save_ltm(sim_state_success, sim_transitions)

# Verify updates in LTM
verified_profile = load_user_profile()
assert verified_profile is not None, "Failed to load profile"
assert len(verified_profile["session_history"]) == 1, "Failed to save session history"
assert verified_profile["session_history"][0]["session_id"] == "session_concluder_verify", "Session id mismatch"
assert verified_profile["session_history"][0]["outcome"] == "successful", "Outcome mismatch"

# Verify Baseline adjustment (Genre should change from 'Rabindra Sangeet' to 'Modern Acoustic' based on positive shift)
assert verified_profile["baseline"]["preferred_genre"] == "Modern Acoustic", f"Expected preferred genre to update to 'Modern Acoustic', got '{verified_profile['baseline']['preferred_genre']}'"
assert verified_profile["baseline"]["total_sessions"] == 1, "Metrics update failed"
assert verified_profile["baseline"]["success_rate"] == 1.0, "Metrics update failed"
print("✅ Test 2: conclude_session_and_save_ltm successfully logs metadata, updates success rates, and adjusts baseline preferred genre.")

print("\n🎉 All Phase 3 Task 4 Verification Tests Passed Successfully!")


--- Running Session Concluder & LTM Verification Tests ---
✅ Test 1: tailored grounding exercise prompts generated correctly for all ending coordinates.
✅ Created new user profile for user_01.
✅ Successfully saved session session_concluder_verify to LTM.
📊 [LTM Baseline Adjustment]: Updated preferred genre from 'Rabindra Sangeet' to 'Modern Acoustic' based on positive shift.
✅ Test 2: conclude_session_and_save_ltm successfully logs metadata, updates success rates, and adjusts baseline preferred genre.

🎉 All Phase 3 Task 4 Verification Tests Passed Successfully!


## 🧪 15. Phase 4: Automated Verification Tests (1.1 - 1.4)
This section implements the automated verification tests for Phase 4 Task 1, covering:
- **1.1 Test Trajectory Calculation & Constraints** (interpolation, clamping, fatigue filter)
- **1.2 Test Target Coordinate Configurations** (Calm and Study Focus targets)
- **1.3 Test Safety Guard Interception** (English, Bengali, and Benglish crisis triggers)
- **1.4 Test LTM Read/Write Integrity** (JSON profile load/save and history logging)

In [22]:
# Phase 4: Automated Verification Tests (1.1 - 1.4)
print("=== Phase 4: Automated Verification Tests ===")

# ----------------------------------------------------
# 1.1 Test Trajectory Calculation & Constraints
# ----------------------------------------------------
print("\n--- Running Test 1.1: Trajectory Calculation & Constraints ---")

# Test correct interpolation math
start = (-0.6, 0.8)
target = (0.7, -0.7)
traj = generate_trajectory(start, target, steps=3)
expected_traj = [(-0.6, 0.8), (0.05, 0.05), (0.7, -0.7)]
assert traj == expected_traj, f"Math check failed: expected {expected_traj}, got {traj}"
print("✅ Interpolation math verified.")

# Test boundary clamping
extreme_start = (-1.5, 2.0)
extreme_target = (1.8, -2.5)
traj_extreme = generate_trajectory(extreme_start, extreme_target, steps=3)
for v, a in traj_extreme:
    assert -1.0 <= v <= 1.0, f"Valence boundary constraint violated: {v}"
    assert -1.0 <= a <= 1.0, f"Arousal boundary constraint violated: {a}"
print("✅ Boundary clamping verified.")

# Test fatigue-prevention filtering
track1 = match_track_to_coordinate(0.7, -0.7, df)
assert track1 is not None, "Failed to match a track"
played = {track1['track_id']}
track2 = match_track_to_coordinate(0.7, -0.7, df, played_ids=played)
assert track2 is not None, "Failed to match a second track"
assert track1['track_id'] != track2['track_id'], "Fatigue filter failed: same track matched again"
print("✅ Fatigue-prevention filtering verified.")

# ----------------------------------------------------
# 1.2 Test Target Coordinate Configurations
# ----------------------------------------------------
print("\n--- Running Test 1.2: Target Coordinate Configurations ---")

# Set current_coords in session_state to allow select_session_goal to run
session_state["current_coords"] = (0.0, 0.0)

# Test Calm Goal
select_session_goal("calm")
assert session_state["goal_type"] == "calm", "Failed to set goal to calm"
assert session_state["target_coords"] == (0.75, -0.75), f"Calm target coordinates mismatch: expected (0.75, -0.75), got {session_state['target_coords']}"
print("✅ Calm target configuration (0.75, -0.75) verified.")

# Test Focus/Study Goal
select_session_goal("focus")
assert session_state["goal_type"] == "focus", "Failed to set goal to focus"
assert session_state["target_coords"] == (0.50, -0.20), f"Focus target coordinates mismatch: expected (0.50, -0.20), got {session_state['target_coords']}"

select_session_goal("study")
assert session_state["goal_type"] == "focus", "Failed to set goal to focus via 'study'"
assert session_state["target_coords"] == (0.50, -0.20), f"Study target coordinates mismatch: expected (0.50, -0.20), got {session_state['target_coords']}"
print("✅ Study/Focus target configuration (0.50, -0.20) verified.")

# ----------------------------------------------------
# 1.3 Test Safety Guard Interception
# ----------------------------------------------------
print("\n--- Running Test 1.3: Safety Guard Interception ---")

# Test Bengali/Benglish crisis inputs
res_bengali1 = safety_guard_skill("ami more jabo, bachaar icche nei")
assert res_bengali1["crisis_detected"] is True, "Failed to detect Bengali crisis phrase 1"

res_bengali2 = safety_guard_skill("mon ta bhalo nei, attahatya korbo")
assert res_bengali2["crisis_detected"] is True, "Failed to detect Bengali crisis phrase 2"

# Test English crisis inputs
res_english = safety_guard_skill("I want to end my life, everything is pointless.")
assert res_english["crisis_detected"] is True, "Failed to detect English crisis phrase"

# Test non-crisis input
res_safe = safety_guard_skill("ami Rabindra Sangeet gaan shunte chai")
assert res_safe["crisis_detected"] is False, "False positive triggered on safe Bengali input"
print("✅ Safety Guard crisis triggers verified for English, Bengali, and Benglish.")

# ----------------------------------------------------
# 1.4 Test LTM Read/Write Integrity
# ----------------------------------------------------
print("\n--- Running Test 1.4: LTM Read/Write Integrity ---")

if os.path.exists(PROFILE_PATH):
    os.remove(PROFILE_PATH)

# Verify profile creation and loading
profile_data = create_user_profile(gad2_score=2, preferred_genre="Folk & Baul")
loaded = load_user_profile()
assert loaded is not None, "Failed to load user profile"
assert loaded["user_id"] == "user_01", "User ID mismatch in loaded profile"
assert loaded["baseline"]["gad2_score"] == 2, "GAD-2 score mismatch"
assert loaded["baseline"]["preferred_genre"] == "Folk & Baul", "Preferred genre mismatch"

# Test history update
session_log = {
    "session_id": "test_ltm_integrity",
    "date": datetime.now().isoformat(),
    "starting_mood": "Sad / Gloomy",
    "starting_coords": [-0.7, -0.4],
    "target_coords": [0.75, -0.75],
    "goal_type": "calm",
    "playlist": [{"track_id": "track_01", "title": "Asha", "feedback": "Calmer"}],
    "completed": True,
    "outcome": "successful"
}
add_session_to_history(session_log)
updated = load_user_profile()
assert len(updated["session_history"]) == 1, "Session history count mismatch"
assert updated["session_history"][0]["session_id"] == "test_ltm_integrity", "Session log integrity check failed"
print("✅ LTM read/write and history logging integrity verified.")

print("\n🎉 All Phase 4 Automated Verification Tests (1.1 - 1.4) completed successfully!")


=== Phase 4: Automated Verification Tests ===

--- Running Test 1.1: Trajectory Calculation & Constraints ---
✅ Interpolation math verified.
✅ Boundary clamping verified.
✅ Fatigue-prevention filtering verified.

--- Running Test 1.2: Target Coordinate Configurations ---
✅ Calm target configuration (0.75, -0.75) verified.
✅ Study/Focus target configuration (0.50, -0.20) verified.

--- Running Test 1.3: Safety Guard Interception ---
✅ Safety Guard crisis triggers verified for English, Bengali, and Benglish.

--- Running Test 1.4: LTM Read/Write Integrity ---
✅ Created new user profile for user_01.
✅ Successfully saved session test_ltm_integrity to LTM.
✅ LTM read/write and history logging integrity verified.

🎉 All Phase 4 Automated Verification Tests (1.1 - 1.4) completed successfully!


## 🧪 16. Phase 4: Conduct Manual Loop Dry Runs (2.1 - 2.3)
This section implements the manual/simulated loop dry runs for Phase 4 Task 2, covering:
- **2.1 Simulate Stress-to-Calm Progression** (using mock transitions)
- **2.2 Simulate Excited-to-Focus Progression** (using mock transitions)
- **2.3 Test Recovery and Recalibration Paths** (verifying 'No Change' recalibration and 'Worse' clinical rescue)

In [23]:
def run_manual_session_simulation(goal_type="calm", starting_mood="Angry / Tense", feedback_sequence=None, interactive=False):
    """
    Simulates or interactively runs a full ISO-therapy session loop.
    - goal_type: 'calm' or 'focus'
    - starting_mood: The name of the starting emotional state
    - feedback_sequence: A list of mock feedback mood selections (for automated runs)
    - interactive: If True, uses live terminal inputs instead of mock sequences
    """
    print("="*80)
    print(f"🎬 STARTING SESSION RUN (Goal: {goal_type.upper()} | Start: {starting_mood} | Mode: {'INTERACTIVE' if interactive else 'SIMULATION'})")
    print("="*80)
    
    # 1. Reset Session State
    global session_state
    session_state = {
        "session_id": f"session_sim_{datetime.now().strftime('%M%S')}",
        "user_id": "user_01",
        "baseline_loaded": False,
        "preferred_genre": None,
        "gad2_score": None,
        "current_mood": None,
        "current_coords": None,
        "target_coords": None,
        "goal_type": None,
        "trajectory": None,
        "playlist": [],
        "played_ids": set(),
        "current_step": 0
    }
    
    # Load profile baseline
    check_user_profile()
    if not session_state["baseline_loaded"]:
        run_onboarding(preferred_genre="Rabindra Sangeet", gad2_score=2)
        
    print(f"👤 Profile Loaded: Preferred Genre = '{session_state['preferred_genre']}'")
    
    # 2. Diagnosing Starting Mood
    print("\n--- [Step 1: Onboarding & Diagnostic Mood Check-in] ---")
    if interactive:
        print("Please check in with your current emotional state:")
        start_selection = get_user_feedback()
    else:
        start_selection = {"name": starting_mood, "coords": EMOTION_COORDINATES[starting_mood]}
        print(f"Mocking check-in selection: '{starting_mood}' at {start_selection['coords']}")
        
    # Check safety
    safety_check = safety_guard_skill(start_selection["name"])
    if safety_check["crisis_detected"]:
        print("🚨 CRISIS DETECTED! Intercepting session...")
        print("Helplines:")
        for k, v in safety_check["hotlines"].items():
            print(f"  - {k}: {v}")
        return
        
    set_mood_coordinates(start_selection["name"], start_selection["coords"][0], start_selection["coords"][1])
    print(f"📍 Baseline Coordinates Registered: {session_state['current_coords']}")
    
    # 3. Setting Goal and Generating Trajectory
    print("\n--- [Step 2: Goal Selection & Trajectory Generation] ---")
    select_session_goal(goal_type)
    print(f"🎯 Target Coordinates: {session_state['target_coords']}")
    print(f"📈 Initial ISO Trajectory: {session_state['trajectory']}")
    
    # Pre-populate initial playlist using matched tracks for each step in trajectory
    session_state["playlist"] = []
    session_state["played_ids"] = set()
    for i, coords in enumerate(session_state["trajectory"]):
        matched_track = match_track_to_coordinate(
            valence_target=coords[0],
            arousal_target=coords[1],
            db_df=df,
            preferred_genre=session_state["preferred_genre"],
            played_ids=session_state["played_ids"]
        )
        if matched_track:
            session_state["playlist"].append(matched_track)
            
    # 4. Playback and Feedback Loop
    print("\n--- [Step 3: Playback & Active Feedback Recalibration] ---")
    step_index = 0
    transitions_log = []
    
    while step_index < len(session_state["playlist"]):
        session_state["current_step"] = step_index
        matched_track = session_state["playlist"][step_index]
        session_state["played_ids"].add(matched_track["track_id"])
        
        print(f"\n🎵 Playing Track {step_index + 1}: '{matched_track['title']}' by {matched_track['artist']}")
        print(f"   Genre: {matched_track['genre']} | Valence={matched_track['valence']}, Arousal={matched_track['arousal']}")
        print(f"   YouTube Link: {matched_track['youtube_url']}")
        
        # Check if final step
        if step_index == len(session_state["playlist"]) - 1:
            print("\n🏁 Final track of the trajectory concluded.")
            break
            
        # Get feedback
        print(f"\n💬 Feedback Check-in after Track {step_index + 1}:")
        if interactive:
            feedback_selection = get_user_feedback(previous_mood=session_state["current_mood"])
        else:
            if feedback_sequence and step_index < len(feedback_sequence):
                fb_name = feedback_sequence[step_index]
                feedback_selection = {"name": fb_name, "coords": EMOTION_COORDINATES[fb_name]}
                print(f"Mocking feedback selection: '{fb_name}' at {feedback_selection['coords']}")
            else:
                fb_name = "Calm / Relaxed" if goal_type == "calm" else "Content / At Ease"
                feedback_selection = {"name": fb_name, "coords": EMOTION_COORDINATES[fb_name]}
                print(f"Mocking default feedback: '{fb_name}' at {feedback_selection['coords']}")
                
        # Run safety check on feedback
        fb_safety = safety_guard_skill(feedback_selection["name"])
        if fb_safety["crisis_detected"]:
            print("🚨 CRISIS DETECTED during feedback! Intercepting session...")
            print("Helplines:")
            for k, v in fb_safety["hotlines"].items():
                print(f"  - {k}: {v}")
            return
            
        # Classify transition
        status = classify_emotional_transition(
            prev_coords=session_state["current_coords"],
            new_coords=feedback_selection["coords"],
            prev_name=session_state["current_mood"],
            new_name=feedback_selection["name"]
        )
        print(f"📊 Transition Class: {status.upper()}")
        transitions_log.append({"step": step_index + 1, "genre": matched_track["genre"], "status": status})
        
        # Update current mood
        set_mood_coordinates(feedback_selection["name"], feedback_selection["coords"][0], feedback_selection["coords"][1])
        
        # Recalibrate trajectory
        recal_msg = recalibrate_session_trajectory(session_state, status, df)
        print(f"🔄 Recalibration Action: {recal_msg}")
        
        # Check if Worse rescue occurred
        if status == "worse":
            rescue_track = session_state["playlist"][-1]
            print(f"\n⚠️ CLINICAL RESCUE TRIGGERED!")
            print(f"🛑 Abruptly halting session queue and playing Clinical Rescue Track: '{rescue_track['title']}' by {rescue_track['artist']}")
            print(f"   YouTube Link: {rescue_track['youtube_url']}")
            set_mood_coordinates("Calm / Relaxed", rescue_track["valence"], rescue_track["arousal"])
            break
            
        step_index += 1
        
    # 5. Session Conclusion & Grounding
    print("\n--- [Step 4: Session Conclusion & Grounding] ---")
    outcome = "successful" if "worse" not in [t["status"] for t in transitions_log] else "rescued"
    sim_summary = {
        "session_id": session_state["session_id"],
        "initial_coords": list(start_selection["coords"]),
        "current_coords": list(session_state["current_coords"]),
        "goal_type": session_state["goal_type"],
        "played_ids": list(session_state["played_ids"]),
        "crisis_flagged": outcome == "rescued"
    }
    
    grounding_msg = conclude_session_and_save_ltm(sim_summary, transitions_log)
    print(grounding_msg)
    print("="*80)
    print(f"🏁 END OF SESSION RUN ({outcome.upper()})")
    print("="*80 + "\n")

# ----------------------------------------------------
# 2.1 Simulate Stress-to-Calm Progression
# ----------------------------------------------------
print("\n=== Running 2.1: Stress-to-Calm Progression Simulation ===")
run_manual_session_simulation(
    goal_type="calm",
    starting_mood="Angry / Tense",
    feedback_sequence=["Sad / Gloomy", "Calm / Relaxed"]
)

# ----------------------------------------------------
# 2.2 Simulate Excited-to-Focus Progression
# ----------------------------------------------------
print("\n=== Running 2.2: Excited-to-Focus Progression Simulation ===")
run_manual_session_simulation(
    goal_type="focus",
    starting_mood="Joyous / Excited",
    feedback_sequence=["Content / At Ease", "Calm / Relaxed"]
)

# ----------------------------------------------------
# 2.3 Test Recovery and Recalibration Paths
# ----------------------------------------------------
print("\n=== Running 2.3: Recalibration Paths Simulations ===")

print("\n👉 2.3(a) No-Change Recalibration Sub-scenario:")
run_manual_session_simulation(
    goal_type="calm",
    starting_mood="Angry / Tense",
    feedback_sequence=["Angry / Tense", "Calm / Relaxed"]
)

print("\n👉 2.3(b) Worse State Rescue Interception Sub-scenario:")
run_manual_session_simulation(
    goal_type="calm",
    starting_mood="Sad / Gloomy",
    feedback_sequence=["Angry / Tense"]
)



=== Running 2.1: Stress-to-Calm Progression Simulation ===
🎬 STARTING SESSION RUN (Goal: CALM | Start: Angry / Tense | Mode: SIMULATION)
👤 Profile Loaded: Preferred Genre = 'Folk & Baul'

--- [Step 1: Onboarding & Diagnostic Mood Check-in] ---
Mocking check-in selection: 'Angry / Tense' at (-0.7, 0.6)
📍 Baseline Coordinates Registered: (-0.7, 0.6)

--- [Step 2: Goal Selection & Trajectory Generation] ---
🎯 Target Coordinates: (0.75, -0.75)
📈 Initial ISO Trajectory: [(-0.7, 0.6), (0.025, -0.075), (0.75, -0.75)]

--- [Step 3: Playback & Active Feedback Recalibration] ---

🎵 Playing Track 1: 'Trishula' by Fossils
   Genre: Bengali Band | Valence=-0.75, Arousal=0.75
   YouTube Link: https://www.youtube.com/watch?v=9_t4E8YtO4g

💬 Feedback Check-in after Track 1:
Mocking feedback selection: 'Sad / Gloomy' at (-0.7, -0.4)
📊 Transition Class: CALMER
🔄 Recalibration Action: Proceeding along the planned therapeutic trajectory as the user is responding positively (Calmer).

🎵 Playing Track 2: 'P

## 🌐 17. Phase 4: Setup ADK Web UI & Browser Integration (3.1 - 3.3)
This section covers the configuration and browser-based deployment instructions for the clinical agents using the Google Agent Development Kit (ADK) Web interface:

### 3.1 Configure Agent Manifests for Web client integration
The CoordinatorAgent and DiagnosticAgent have been packaged into the `music_therapy_agent` python package. In the package entrypoint, we expose `root_agent = CoordinatorAgent`. This allows the ADK framework to automatically discover and orchestrate our system.

### 3.2 Run the ADK Web UI local server
To start the local ADK Web UI server, execute `adk web` from the workspace root directory. Below, you can run the setup cell to print guidelines or start the server directly.

In [24]:
# Google ADK Web UI Setup Instructions
print("========================================================================")
print("         LAUNCHING BROWSER INTEGRATION (Google ADK Web UI)              ")
print("========================================================================")
print("1. The agents are packaged inside the 'music_therapy_agent/' package.")
print("2. To run the Web interface, start a terminal at the project directory.")
print("3. Execute the following CLI command inside your virtual environment:")
print("      adk web")
print("4. Once started, open your web browser and navigate to:")
print("      http://localhost:8080")
print("5. In the top-left dropdown menu, select 'CoordinatorAgent' and begin your session.")
print("========================================================================")


         LAUNCHING BROWSER INTEGRATION (Google ADK Web UI)              
1. The agents are packaged inside the 'music_therapy_agent/' package.
2. To run the Web interface, start a terminal at the project directory.
3. Execute the following CLI command inside your virtual environment:
      adk web
4. Once started, open your web browser and navigate to:
      http://localhost:8080
5. In the top-left dropdown menu, select 'CoordinatorAgent' and begin your session.
